# Topic 30 · Remember the first symbol: RNN against LSTM

CSCI 6379 · companion to the lecture note [Recurrent neural networks](https://dongchul.kim/csci6379-fall2026/topic/30).

A sequence starts with A, B, C or D, then random filler. At the end the network must say which symbol it started with. We train a plain RNN, an LSTM as PyTorch creates it, and an LSTM whose **forget gate starts open**, at several lengths. Then we measure how much gradient reaches each step.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. It takes about 10 minutes with one seed on a T4.

In [ ]:
import time
import torch, torch.nn as nn
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
HID = 64

def batch(T, n, g):
    first = torch.randint(0, 4, (n,), generator=g)              # the signal: 0..3 (A..D)
    rest = torch.randint(4, 8, (n, T - 1), generator=g)          # filler: 4..7, random
    return torch.cat([first[:, None], rest], 1).to(device), first.to(device)

class Net(nn.Module):
    def __init__(self, kind, forget_bias=None):
        super().__init__()
        self.emb = nn.Embedding(8, 16)
        self.rnn = (nn.LSTM if kind == "lstm" else nn.RNN)(16, HID, batch_first=True)
        self.out = nn.Linear(HID, 4)
        if forget_bias is not None:      # PyTorch orders an LSTM's gates input, forget, cell, output
            with torch.no_grad():
                for name, p in self.rnn.named_parameters():
                    if "bias" in name:
                        p[HID:2 * HID].fill_(forget_bias / 2)    # two bias vectors that add up
    def forward(self, x):
        h, _ = self.rnn(self.emb(x))
        return self.out(h[:, -1])        # many-to-one: answer after the last symbol

In [ ]:
def run(kind, forget_bias, T, seed=0, steps=3000):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(1000 + seed)
    net = Net(kind, forget_bias).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3); loss_fn = nn.CrossEntropyLoss()
    for s in range(steps):
        x, y = batch(T, 128, g)
        loss = loss_fn(net(x), y)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
    with torch.no_grad():
        x, y = batch(T, 4000, torch.Generator().manual_seed(9999))
        return (net(x).argmax(1) == y).float().mean().item()

MODELS = {"plain RNN": ("rnn", None), "LSTM, as PyTorch starts it": ("lstm", None),
          "LSTM, forget gate open": ("lstm", 3.0)}
Ts = [5, 10, 20, 50, 100, 200]
acc = {m: [] for m in MODELS}
t0 = time.time()
for m, (kind, fb) in MODELS.items():
    for T in Ts:
        acc[m].append(run(kind, fb, T))
        print(f"{m:28s} T={T:3d}  accuracy {acc[m][-1]:.2f}  ({time.time() - t0:.0f}s)")

plt.figure(figsize=(6, 3.6))
for m in MODELS:
    plt.plot(Ts, acc[m], "o-", label=m)
plt.axhline(0.25, color="gray", ls="--"); plt.xscale("log"); plt.xticks(Ts, Ts)
plt.xlabel("sequence length T"); plt.ylabel("test accuracy"); plt.legend(); plt.show()

## The vanishing gradient, measured

Untrained networks, 100 steps: how much gradient reaches each input step? (Logarithmic axis.)

In [ ]:
torch.set_default_dtype(torch.float64)        # the small numbers need double precision
plt.figure(figsize=(7, 3.6))
for m, (kind, fb) in MODELS.items():
    torch.manual_seed(0)
    cell = (nn.LSTM if kind == "lstm" else nn.RNN)(16, HID, batch_first=True)
    if fb is not None:
        with torch.no_grad():
            for name, p in cell.named_parameters():
                if "bias" in name: p[HID:2 * HID].fill_(fb / 2)
    x = torch.randn(256, 100, 16, requires_grad=True)
    h, _ = cell(x)
    nn.functional.cross_entropy(nn.Linear(HID, 4)(h[:, -1]), torch.randint(0, 4, (256,))).backward()
    g = x.grad.norm(dim=2).mean(0)
    plt.plot(range(1, 101), g.numpy(), label=m)
    print(f"{m:28s} gradient at step 1 / at step 100 = {(g[0] / g[-1]).item():.1e}")
plt.yscale("log"); plt.xlabel("input step"); plt.ylabel("gradient reaching it"); plt.legend(); plt.show()
torch.set_default_dtype(torch.float32)